# 01: Why Measurement Matters

Module 1 starts by treating evaluation as an engineering discipline, not a vibe check. This notebook builds intuition for why anecdotal testing fails, how offline and online evals answer different questions, and how a tiny benchmark harness can compare prompt variants on the same task set.

## Learning goals

- understand why evals are the measurement layer for prompts, RAG, and agents
- distinguish offline evals from online product metrics
- see why benchmark design matters as much as scoring
- build a small benchmark harness from scratch
- compare prompt variants on the same labeled task set

In [ ]:
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes torch

import csv
import json
import math
import random
import statistics
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-14B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True, top_k=20):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=top_k,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Why measurement changes the way you build

Without evals, improvement claims usually sound like this:

- this prompt felt better on the examples I tried
- the new version seemed more careful
- the model looked smarter in the demo

Those statements may be sincere, but they are not enough for engineering. Evals force you to define:

- the task
- the target behavior
- the dataset
- the scorer
- the comparison rule

That turns model iteration from storytelling into measurement.

## Anecdotes are not evidence

Anecdotal testing is useful for exploration, but dangerous for decision-making. It tends to:

- over-sample easy examples
- ignore rare but costly failures
- overweight the last run you saw
- reward prompts that look polished instead of prompts that are consistently correct

We can show this with a tiny made-up story before we build a real benchmark.

In [ ]:

def print_table(rows, headers=None):
    rows = list(rows)
    if not rows:
        print("No rows")
        return

    if headers is None:
        if isinstance(rows[0], dict):
            headers = list(rows[0].keys())
        else:
            headers = [f"col_{index}" for index in range(len(rows[0]))]

    normalized = []
    for row in rows:
        if isinstance(row, dict):
            normalized.append([str(row.get(header, "")) for header in headers])
        else:
            normalized.append([str(value) for value in row])

    widths = []
    for index, header in enumerate(headers):
        content_width = max(len(values[index]) for values in normalized)
        widths.append(max(len(str(header)), content_width))

    header_line = " | ".join(str(header).ljust(widths[index]) for index, header in enumerate(headers))
    divider = "-+-".join("-" * width for width in widths)

    print(header_line)
    print(divider)
    for values in normalized:
        print(" | ".join(values[index].ljust(widths[index]) for index in range(len(headers))))


In [ ]:

prompt_trials = [
    {"variant": "prompt_a", "ticket": "I was double charged for my subscription.", "expected": "billing", "prediction": "billing", "looked_good": True},
    {"variant": "prompt_a", "ticket": "The export button keeps failing with a timeout.", "expected": "technical_issue", "prediction": "technical_issue", "looked_good": True},
    {"variant": "prompt_a", "ticket": "Can you add Slack alerts to weekly reports?", "expected": "feature_request", "prediction": "feature_request", "looked_good": True},
    {"variant": "prompt_a", "ticket": "Our intern can see payroll files after being offboarded.", "expected": "security", "prediction": "account_access", "looked_good": False},
    {"variant": "prompt_a", "ticket": "I cannot sign in after resetting my password twice.", "expected": "account_access", "prediction": "technical_issue", "looked_good": False},
]

anecdote_view = [row for row in prompt_trials if row["looked_good"]]
anecdote_accuracy = sum(row["prediction"] == row["expected"] for row in anecdote_view) / len(anecdote_view)
full_accuracy = sum(row["prediction"] == row["expected"] for row in prompt_trials) / len(prompt_trials)

print("Anecdote-only accuracy:", round(anecdote_accuracy, 3))
print("Full task-set accuracy:", round(full_accuracy, 3))
print()
print_table(prompt_trials, headers=["ticket", "expected", "prediction", "looked_good"])


## Offline versus online evals

Offline evals measure behavior on a fixed task set with known labels. They are best for:

- prompt comparisons
- ablations
- regression checks
- fast iteration before release

Online evals measure behavior in the product. They are best for:

- user satisfaction
- task completion
- escalation rate
- retention or conversion impact

Neither replaces the other. Offline evals tell you if a change looks promising. Online evals tell you if that change survives contact with real users.

In [ ]:

offline_online = [
    {
        "question": "What is measured?",
        "offline_eval": "Quality on a labeled benchmark",
        "online_eval": "Behavior in a live product"
    },
    {
        "question": "When is it useful?",
        "offline_eval": "Before release and during iteration",
        "online_eval": "After release under real traffic"
    },
    {
        "question": "Main strength",
        "offline_eval": "Cheap controlled comparisons",
        "online_eval": "Captures real-world value"
    },
    {
        "question": "Main risk",
        "offline_eval": "May miss distribution shift",
        "online_eval": "Harder to attribute causes"
    },
]

print_table(offline_online, headers=["question", "offline_eval", "online_eval"])


## What a benchmark needs

A benchmark is not just a pile of examples. A useful benchmark needs:

1. a clear task definition
2. realistic examples
3. labels or grading criteria
4. a scorer
5. a comparison rule
6. enough variety that a lucky streak does not fool you

We will use a small support-ticket routing task because it is concrete, structured, and easy to score.

In [ ]:

benchmark = [
    {"id": "t1", "ticket": "I was charged twice after upgrading our plan.", "label": "billing", "difficulty": "easy"},
    {"id": "t2", "ticket": "The dashboard freezes whenever I open the team analytics tab.", "label": "technical_issue", "difficulty": "easy"},
    {"id": "t3", "ticket": "Please add an option to export charts directly to PowerPoint.", "label": "feature_request", "difficulty": "easy"},
    {"id": "t4", "ticket": "A former contractor can still view private project folders.", "label": "security", "difficulty": "medium"},
    {"id": "t5", "ticket": "Password reset emails never arrive for two of our managers.", "label": "account_access", "difficulty": "medium"},
    {"id": "t6", "ticket": "The invoice says annual billing even though we switched to monthly.", "label": "billing", "difficulty": "medium"},
    {"id": "t7", "ticket": "Mobile app uploads fail only on large video files.", "label": "technical_issue", "difficulty": "medium"},
    {"id": "t8", "ticket": "Can you support approval workflows for procurement requests?", "label": "feature_request", "difficulty": "medium"},
    {"id": "t9", "ticket": "We saw repeated login attempts from an unknown location overnight.", "label": "security", "difficulty": "hard"},
    {"id": "t10", "ticket": "Single sign-on stopped working after our identity provider certificate rotated.", "label": "account_access", "difficulty": "hard"},
]

print("Benchmark size:", len(benchmark))
print_table(benchmark[:6], headers=["id", "label", "difficulty", "ticket"])


## Check the task mix before you trust the results

If your dataset is 90 percent easy billing tickets, a prompt can look strong while still failing on security or access issues. Simple distribution checks often catch benchmark design mistakes early.

In [ ]:

from collections import Counter

label_counts = Counter(task["label"] for task in benchmark)
difficulty_counts = Counter(task["difficulty"] for task in benchmark)

label_rows = [{"label": label, "count": count} for label, count in sorted(label_counts.items())]
difficulty_rows = [{"difficulty": level, "count": count} for level, count in sorted(difficulty_counts.items())]

print("Label distribution")
print_table(label_rows)
print()
print("Difficulty distribution")
print_table(difficulty_rows)


## Prompt variants are hypotheses

An eval compares alternatives under the same conditions. Here we will test three prompt variants:

- a direct instruction
- a rule-first instruction
- a few-shot prompt

The goal is not to find the perfect prompt. The goal is to make comparison disciplined.

In [ ]:

PROMPT_VARIANTS = {
    "direct_label": """You classify support tickets.
Return exactly one label from: {labels}
Do not explain your answer.

Ticket:
{ticket}""",
    "rule_first": """You classify support tickets into one of these labels: {labels}

Rules:
- billing is about charges, invoices, or plan changes
- technical_issue is about bugs, crashes, or broken product behavior
- feature_request is about asking for a new capability
- security is about unauthorized access, suspicious activity, or exposure risk
- account_access is about login, password reset, SSO, or account recovery

Return only the label.

Ticket:
{ticket}""",
    "few_shot": """You classify support tickets.
Return exactly one label from: {labels}

Examples:
Ticket: We need a refund because the renewal billed the wrong card.
Label: billing

Ticket: The page crashes whenever we attach a PDF.
Label: technical_issue

Ticket: Please add support for Okta group sync.
Label: feature_request

Ticket: A user who left last month still has access to customer records.
Label: security

Ticket: Nobody can sign in after we enforced two-factor authentication.
Label: account_access

Now classify this ticket.
Ticket:
{ticket}

Label:""",
}

print("Prompt variants:", list(PROMPT_VARIANTS.keys()))


## Build a tiny benchmark harness

The harness needs four pieces:

- prompt construction
- prediction normalization
- scoring
- experiment aggregation

We will keep the scorer intentionally simple: exact match on the label.

In [ ]:

VALID_LABELS = sorted({task["label"] for task in benchmark})

def build_prompt(variant_name, ticket_text):
    return PROMPT_VARIANTS[variant_name].format(
        labels=", ".join(VALID_LABELS),
        ticket=ticket_text,
    )

def normalize_label(text):
    cleaned = text.strip().lower()
    if cleaned.startswith("label:"):
        cleaned = cleaned.split(":", 1)[1].strip()
    cleaned = cleaned.splitlines()[0].strip()
    cleaned = cleaned.replace("-", "_").replace(" ", "_")
    while cleaned and not (cleaned[-1].isalnum() or cleaned[-1] == "_"):
        cleaned = cleaned[:-1]
    return cleaned

def classify_ticket(ticket_text, variant_name):
    prompt = build_prompt(variant_name, ticket_text)
    raw = generate(prompt, max_new_tokens=16, temperature=0.0, do_sample=False)
    prediction = normalize_label(raw)
    if prediction not in VALID_LABELS:
        prediction = f"unrecognized:{prediction}"
    return raw, prediction

def evaluate_variant(tasks, variant_name):
    rows = []
    for task in tasks:
        raw, prediction = classify_ticket(task["ticket"], variant_name)
        passed = int(prediction == task["label"])
        rows.append(
            {
                "id": task["id"],
                "variant": variant_name,
                "gold": task["label"],
                "prediction": prediction,
                "passed": passed,
                "raw_output": raw,
            }
        )

    accuracy = sum(row["passed"] for row in rows) / len(rows)
    return rows, accuracy


## Run the benchmark on a small task slice

For a first pass, keep the experiment small and cheap. A tiny slice is enough to reveal obvious prompt differences before you scale up.

In [ ]:

eval_slice = benchmark[:6]
experiment_rows = []
leaderboard = []

for variant_name in PROMPT_VARIANTS:
    rows, accuracy = evaluate_variant(eval_slice, variant_name)
    experiment_rows.extend(rows)
    leaderboard.append(
        {
            "variant": variant_name,
            "accuracy": round(accuracy, 3),
            "correct": sum(row["passed"] for row in rows),
            "total": len(rows),
        }
    )

print("Completed runs:", len(experiment_rows))
print_table(leaderboard, headers=["variant", "accuracy", "correct", "total"])


## Compare prompt variants

The leaderboard is useful, but only after you verify that:

- each variant saw the same tasks
- scoring was consistent
- outputs were normalized the same way

Those details sound small, but they are where accidental unfairness often enters.

In [ ]:

leaderboard_sorted = sorted(leaderboard, key=lambda row: row["accuracy"], reverse=True)
print_table(leaderboard_sorted, headers=["variant", "accuracy", "correct", "total"])


## Look at per-example behavior

Averages hide disagreement. Per-example comparison shows which tickets are robustly easy and which ones are sensitive to prompt phrasing.

In [ ]:

comparison_rows = []
for task in eval_slice:
    row = {"id": task["id"], "gold": task["label"]}
    for variant_name in PROMPT_VARIANTS:
        match = next(
            result for result in experiment_rows
            if result["id"] == task["id"] and result["variant"] == variant_name
        )
        row[variant_name] = match["prediction"]
    comparison_rows.append(row)

print_table(comparison_rows, headers=["id", "gold", "direct_label", "rule_first", "few_shot"])


## Inspect the misses, not just the average

Error analysis starts immediately. A good eval loop does not stop at the score; it asks what pattern sits behind the misses.

In [ ]:

error_rows = [row for row in experiment_rows if row["passed"] == 0]

if error_rows:
    print_table(error_rows, headers=["id", "variant", "gold", "prediction", "raw_output"])
else:
    print("No errors on this slice. Add harder cases before trusting the result.")


## Save artifacts for later comparison

Even tiny experiments should leave behind artifacts. That makes later comparisons reproducible and lets you inspect a run without rerunning the model.

In [ ]:

artifact_dir = Path("module_1_artifacts")
artifact_dir.mkdir(exist_ok=True)

leaderboard_path = artifact_dir / "01_prompt_variant_leaderboard.json"
rows_path = artifact_dir / "01_prompt_variant_rows.json"

with leaderboard_path.open("w", encoding="utf-8") as handle:
    json.dump(leaderboard_sorted, handle, indent=2)

with rows_path.open("w", encoding="utf-8") as handle:
    json.dump(experiment_rows, handle, indent=2)

print("Saved:", leaderboard_path)
print("Saved:", rows_path)


## Benchmark design checklist

Before you trust an eval result, ask:

- does the dataset reflect the task distribution you care about
- are expensive failure modes represented
- is the scorer aligned with business value
- are prompt variants compared on identical examples
- can someone else rerun the experiment and get the same result

In [ ]:

design_principles = [
    {"principle": "Representative data", "why_it_matters": "Prevents wins that disappear on real traffic"},
    {"principle": "Clear labels", "why_it_matters": "Makes scoring unambiguous"},
    {"principle": "Consistent scoring", "why_it_matters": "Keeps comparisons fair"},
    {"principle": "Saved artifacts", "why_it_matters": "Supports reproducibility and audit"},
    {"principle": "Error review", "why_it_matters": "Turns scores into next actions"},
]

print_table(design_principles, headers=["principle", "why_it_matters"])


## Offline results do not replace online learning

Imagine the best offline prompt also reduced escalation rate, while another prompt produced slightly nicer wording but more wrong routes. Product decisions need both views.

In [ ]:

simulated_online_metrics = [
    {"variant": "rule_first", "offline_accuracy": next(row["accuracy"] for row in leaderboard if row["variant"] == "rule_first"), "ticket_escalation_rate": 0.12, "user_follow_up_rate": 0.18},
    {"variant": "direct_label", "offline_accuracy": next(row["accuracy"] for row in leaderboard if row["variant"] == "direct_label"), "ticket_escalation_rate": 0.15, "user_follow_up_rate": 0.21},
    {"variant": "few_shot", "offline_accuracy": next(row["accuracy"] for row in leaderboard if row["variant"] == "few_shot"), "ticket_escalation_rate": 0.14, "user_follow_up_rate": 0.19},
]

print_table(simulated_online_metrics, headers=["variant", "offline_accuracy", "ticket_escalation_rate", "user_follow_up_rate"])


## Takeaways

- anecdotal testing is useful for discovery, not for trustworthy comparison
- offline evals make iteration cheap and disciplined
- online evals measure real user impact
- benchmark design is part of the result, not a detail outside the result
- even a tiny harness can already compare prompt variants, save artifacts, and surface error patterns

In the next notebook, we will focus on the dataset itself: how to build a golden set that is realistic, inspectable, and hard to accidentally contaminate.